# NB20 — Aggregating the seeds

Reads the per-seed score files from NB19 together with the seed-42 scores from NB15, applies the same validity gate as NB16 (garbled and no-op removed), and reports the hybrid minus neural gap **per seed** and **pooled across seeds**, with the paired bootstrap.

This notebook is CPU-only and takes a few minutes; it is kept separate from NB19 so that it does not consume GPU quota.

**What it answers.** Whether the −2.93 point effects in the thesis survive retraining, or whether they lie inside run-to-run variation.

In [ ]:
# ============================ 1/6  inputs ============================
import numpy as np, pandas as pd, itertools

# seed 42 comes from the original run; the others from NB19
P_SEED42 = "/kaggle/input/notebooks/bahaaqassem/nb15-stress-evaluation/nb15_stress_results.parquet"
P_SEED_N = ["/kaggle/input/notebooks/bahaaqassem/nb19-seed-replication/nb19_seed1_scores.parquet",
            "/kaggle/input/notebooks/bahaaqassem/nb19-seed-replication/nb19_seed2_scores.parquet"]
# the validity fields computed in NB16 (cosine, jaccard, length ratio) keyed by gen_id
P_VALIDITY = "/kaggle/input/notebooks/bahaaqassem/nb16-prep-bootstrap/stress_generations_meta.parquet"

N_BOOT = 2000
RNG    = np.random.default_rng(42)
print("[1/6] configuration read", flush=True)

In [ ]:
# ============================ 2/6  load every seed into one frame ============================
frames = []
s42 = pd.read_parquet(P_SEED42)
keep = ["gen_id", "article_id", "model", "task", "level",
        "p_hybrid_base", "p_hybrid_gen", "p_neural_base", "p_neural_gen"]
s42 = s42[[c for c in keep if c in s42.columns]].copy()
s42["seed"] = 42
frames.append(s42)
print(f"[2/6] seed 42: {len(s42)} rows", flush=True)
for p in P_SEED_N:
    d = pd.read_parquet(p)
    frames.append(d[[c for c in keep + ["seed"] if c in d.columns]])
    print(f"[2/6] {p.split('/')[-1]}: {len(d)} rows, seed {int(d.seed.iloc[0])}", flush=True)
S = pd.concat(frames, ignore_index=True)
print(f"[2/6] combined: {len(S)} rows | seeds {sorted(S.seed.unique())}", flush=True)
assert S.groupby("seed").size().nunique() == 1, "seeds cover different numbers of generations"

In [ ]:
# ============================ 3/6  the NB16 validity gate ============================
V = pd.read_parquet(P_VALIDITY)
cols = [c for c in ("gen_id", "cosine", "jaccard", "len_ratio", "valid") if c in V.columns]
V = V[cols].drop_duplicates("gen_id")
S = S.merge(V, on="gen_id", how="left")
if "valid" not in S.columns:
    S["garbled"] = (S.cosine <= 0.5) | (~S.len_ratio.between(0.6, 1.6))
    S["noop"]    = S.jaccard > 0.98
    S["valid"]   = ~(S.garbled | S.noop)
    print("[3/6] validity recomputed from cosine, jaccard and length ratio", flush=True)
else:
    print("[3/6] validity read from the NB16 file", flush=True)
print(f"[3/6] valid rows per seed:\n{S.groupby('seed').valid.sum().to_string()}", flush=True)
S = S[S.valid].copy()
print(f"[3/6] kept {len(S)} rows after the gate", flush=True)

In [ ]:
# ============================ 4/6  rates per seed ============================
def rate(p): return 100.0*float((np.asarray(p) >= 0.5).mean()) if len(p) else float("nan")

rows = []
for seed, s in S.groupby("seed"):
    pol = s[s.task == "polish"]; hum = s[s.task == "humanize"]
    rows.append({
        "seed": int(seed), "n_polish": len(pol), "n_humanize": len(hum),
        "FPR_hyb": round(rate(pol.p_hybrid_gen), 2), "FPR_neu": round(rate(pol.p_neural_gen), 2),
        "TPR_hyb": round(rate(hum.p_hybrid_gen), 2), "TPR_neu": round(rate(hum.p_neural_gen), 2),
        "ctrl_FPR_hyb": round(rate(pol.p_hybrid_base), 2), "ctrl_FPR_neu": round(rate(pol.p_neural_base), 2),
    })
R = pd.DataFrame(rows)
R["gap_FPR"] = (R.FPR_hyb - R.FPR_neu).round(2)
R["gap_TPR"] = (R.TPR_hyb - R.TPR_neu).round(2)
print("[4/6] per-seed rates, hybrid against neural", flush=True)
print(R.to_string(index=False), flush=True)
print(f"\n[4/6] gap on the false-alarm axis: mean {R.gap_FPR.mean():+.2f} "
      f"sd {R.gap_FPR.std(ddof=1):.2f} range [{R.gap_FPR.min():+.2f}, {R.gap_FPR.max():+.2f}]", flush=True)
print(f"[4/6] gap on the evasion axis:     mean {R.gap_TPR.mean():+.2f} "
      f"sd {R.gap_TPR.std(ddof=1):.2f} range [{R.gap_TPR.min():+.2f}, {R.gap_TPR.max():+.2f}]", flush=True)

In [ ]:
# ============================ 5/6  paired bootstrap, per seed and pooled ============================
def boot_gap(sub, col_h, col_n, label):
    h = (sub[col_h].to_numpy() >= 0.5).astype(float)
    n = (sub[col_n].to_numpy() >= 0.5).astype(float)
    if len(h) == 0: return None
    obs = 100.0*(h.mean() - n.mean())
    idx = RNG.integers(0, len(h), size=(N_BOOT, len(h)))
    d = 100.0*(h[idx].mean(1) - n[idx].mean(1))
    lo, hi = np.percentile(d, [2.5, 97.5])
    sig = "yes" if (lo > 0) or (hi < 0) else "no"
    print(f"[5/6] {label:34s} gap {obs:+6.2f}  95% CI [{lo:+6.2f}, {hi:+6.2f}]  significant {sig}", flush=True)
    return {"scope": label, "n": len(h), "gap": round(obs, 2),
            "lo": round(float(lo), 2), "hi": round(float(hi), 2), "significant": sig}

res = []
for seed, s in S.groupby("seed"):
    pol = s[s.task == "polish"]; hum = s[s.task == "humanize"]
    res.append(boot_gap(pol, "p_hybrid_gen", "p_neural_gen", f"seed {int(seed)} · polishing (FPR)"))
    res.append(boot_gap(hum, "p_hybrid_gen", "p_neural_gen", f"seed {int(seed)} · humanizing (TPR)"))
    res.append(boot_gap(pol, "p_hybrid_base", "p_neural_base", f"seed {int(seed)} · control, untransformed"))
print("", flush=True)
pol = S[S.task == "polish"]; hum = S[S.task == "humanize"]
res.append(boot_gap(pol, "p_hybrid_gen", "p_neural_gen", "all seeds pooled · polishing"))
res.append(boot_gap(hum, "p_hybrid_gen", "p_neural_gen", "all seeds pooled · humanizing"))
res.append(boot_gap(pol, "p_hybrid_base", "p_neural_base", "all seeds pooled · control"))
B = pd.DataFrame([r for r in res if r])

In [ ]:
# ============================ 6/6  verdict and outputs ============================
R.to_parquet("/kaggle/working/nb20_per_seed_rates.parquet", index=False)
B.to_parquet("/kaggle/working/nb20_bootstrap.parquet", index=False)
S.to_parquet("/kaggle/working/nb20_all_seed_scores.parquet", index=False)
print("[6/6] wrote nb20_per_seed_rates.parquet, nb20_bootstrap.parquet, nb20_all_seed_scores.parquet", flush=True)

same_sign_fpr = bool((R.gap_FPR < 0).all() or (R.gap_FPR > 0).all())
same_sign_tpr = bool((R.gap_TPR < 0).all() or (R.gap_TPR > 0).all())
sig_all = [r for r in B.itertuples() if r.scope.startswith("seed") and "control" not in r.scope]
n_sig = sum(1 for r in sig_all if r.significant == "yes")
print("", flush=True)
print("[6/6] READ THIS AS FOLLOWS", flush=True)
print(f"[6/6]   the false-alarm gap keeps its sign across every seed: {same_sign_fpr}", flush=True)
print(f"[6/6]   the evasion gap keeps its sign across every seed:     {same_sign_tpr}", flush=True)
print(f"[6/6]   per-seed effects significant: {n_sig} of {len(sig_all)}", flush=True)
print(f"[6/6]   spread between seeds, false alarm {R.gap_FPR.max()-R.gap_FPR.min():.2f} points, "
      f"evasion {R.gap_TPR.max()-R.gap_TPR.min():.2f} points", flush=True)
print("[6/6]   if the between-seed spread is of the same order as the effect itself,", flush=True)
print("[6/6]   the thesis must report the effect as seed-dependent rather than established.", flush=True)
print(B.to_string(index=False), flush=True)